<div style="text-align: justify; max-width: 90ch">

# Lecture 03 exercises

Exercises are tagged with the goal they practise (see `reading_material/goals_03.md`). Time bands:

- 15 min (quick)
- 45 min (realistic)
- 90 min

Before you start:

- Run the configuration cell first, then the **Rebuild inputs** cell.
- Ollama must be running with the models pulled.
- The MLflow server must be started (guide `02a_mlflow_server`).
- `lec_02d` must have run once: the exercises reuse its Chroma index (otherwise the Rebuild inputs cell builds it, which takes minutes).

</div>

In [ ]:
import os
import time
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from mlflow.genai.scorers import scorer

from course_helpers.agents import (
    GRADE_PROMPT,
    GRADE_SCHEMA,
    make_agentic_rag,
    make_course_tools,
    message_table,
    run_tool_calls,
    searches,
    total_tokens,
)
from course_helpers.checks import check_mlflow
from course_helpers.corpus import load_corpus, load_eval_set
from course_helpers.evaluation import (
    DETERMINISTIC_SCORERS,
    compare_variants,
    fact_keywords_found,
    is_refusal,
    judge_input_tokens,
    make_facts_judge,
    per_question_table,
    prompt_tokens,
    build_eval_records_from,
)
from course_helpers.rag import (
    LECTURE_KINDS,
    build_chroma_index,
    format_context_from,
    make_rag_chain,
    split_documents,
)

In [ ]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-exercises"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs** (given, run it once): the system of `lec_03a`-`lec_03b` from the `course_helpers` helpers, the notes as `Document`s for the tools of `lec_03c`,
the evaluation set by question id, and tracing into the experiment `llm-course-03-exercises`. The chat model is that of `lec_03b`:
`num_predict=200` and no `num_ctx`, so it shares Ollama's default window with the judge and the model is not reloaded.

</div>

In [ ]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

documents = load_corpus(REPO / "corpus" / "uoa_py_course")
store = build_chroma_index(
    split_documents(documents, chunk_size=800, chunk_overlap=100),
    "course_notes",
    DATA_DIR / "chroma",
    EMBED_MODEL,
    OLLAMA_HOST,
)
retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=200,
)
rag_chain = make_rag_chain(retriever, llm)

eval_set = load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")
eval_by_id = {row["id"]: row for row in eval_set}

check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

print(f"{len(eval_set)} questions in the evaluation set")

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.1 [G1, G2] All 19 questions, aggregates against rows (45 min)

Turn the whole evaluation set into MLflow records (`build_eval_records_from`) and evaluate the RAG chain of the Rebuild inputs cell on all 19 questions with the deterministic scorers of `lec_03a` (`DETERMINISTIC_SCORERS`),
in one named run. Print the aggregate metrics, then the per-question table (`per_question_table`).

- Which scorer's mean hides the most failing rows?
- For two failing rows, read the answer (in the MLflow UI or in `results.result_df["response"]`) and decide: is the answer wrong, or only the check?

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Which mean hides failing rows, and for two failing rows: is the answer wrong, or only the check?*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.2 [G2] A scorer of your own (15 min)

Write a scorer with `@scorer` for a behaviour of your choice, for example "an out-of-scope question is answered with a refusal":
records of questions the notes do not cover carry an expectation that says so, and the scorer abstains (`None`) on all other records.
Test it first on two answers you write yourself, where you know the right verdict, then run it in an evaluation on three out-of-scope questions of your own.

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Did your scorer give the verdict you expected on every answer? Where not, is the answer or the scorer at fault?*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.3 [G3] The judge against the keywords on all 19 (45 min)

Grade the 19 answers of exercise 3.1 with the facts judge of `lec_03b` (`make_facts_judge`) and with `fact_keywords_found`,
without generating new answers: records that carry `outputs` need no `predict_fn`, so both scorers read exactly the same answers.
List the questions where the two disagree (the keywords find every fact but the judge says `no`, or the other way round) and read the judge's rationale for each.
Finally, check with `judge_input_tokens` that every judge prompt fit Ollama's default window (4,096 tokens unless the GPU has 24 GB of VRAM or more,
`lec_03b` section 4).

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Where the two disagree, who is right: the keywords or the judge? Quote the rationale that convinced you.*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.4 [G4] A variant of your own, quality against cost (45 min)

Pick one change to the system: `num_predict` 200 against 400, or a change to the prompt (`make_rag_chain(retriever, llm, prompt_template=...)`).
Evaluate both variants on all 19 questions with the same suite, the deterministic scorers (`answer_tokens` among them), `prompt_tokens` and the facts judge,
one named run each, and put them side by side with `compare_variants`. Which rows of the table measure quality, and which cost?
Which variant would you ship?

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Which rows measure quality and which cost, and which variant would you ship?*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.5 [G5] A tool of your own (45 min)

Write a read-only tool with `@tool` that the three tools of `lec_03c` lack, for example the number of words of one notebook,
given its file name. It must check its argument and **return** an error the model can read, not raise. `run_tool_calls(reply, tools)` (`course_helpers.agents`) runs every tool call of one model reply and returns one `ToolMessage` per call,
an error message for a bad call. `message_table(messages)` gives one row per message: its role, its tool calls and its text.
Then:

- run the loop of `lec_03c` section 3 by hand for a question your tool answers, and print the conversation with `message_table`;
- write a bad model reply yourself, an `AIMessage` whose `tool_calls` hold a wrong value, a wrong type and an unknown tool name,
  and run it with `run_tool_calls`. What does each error tell the model?

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Did the model call your tool with the right argument? What does each of the three errors tell the model to do next?*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.6 [G6] Thinking on and off for a tool choice (15 min)

Bind the course's tools (`make_course_tools(documents, retriever)`) to the chat model and ask the question of `lec_03c` section 6,
"Do lectures 9 and 13 have the same number of lecture notebooks?", in single calls: five with thinking off and five with thinking on, at the same temperature.
For each call record whether it asks `list_notebooks` for both lectures, its output tokens and its seconds. Does thinking help this decision, and at what price?

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Does thinking help this decision, and what does it cost in tokens and seconds?*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.7 [G8] The chain against the graph on all 19 and your own wording (90 min)

Write three questions of your own that reword questions of the evaluation set the way a student might, without the notes' terms (as `lec_03d` section 1 did for q19),
and give each the expectations of its original. Evaluate the plain chain and the graph of `lec_03d` (`make_agentic_rag`) on the 19 questions plus yours,
with the deterministic scorers, `searches`, `total_tokens` and the facts judge of `lec_03b`, both with `num_predict=300` as in `lec_03d`.
Compare the aggregates, then the rows where the two systems differ. End with a decision record in the format of `lec_03d` section 8:
is the loop worth its price for this course?

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Your decision record: decision, requirements, evidence, worth it when, not worth it when, revisit when.*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.8 [G7, G8] A strict grade against the topic grade (45 min)

`lec_03d` grades the chunks with a question about the **topic** (`GRADE_PROMPT` in `course_helpers.agents`). Write a strict grade prompt that asks whether the notes contain the **answer**,
with the same `GRADE_SCHEMA`, and grade the same searches with both prompts:

- the 19 questions in their own words, where the expected file is found;
- your reworded questions of exercise 3.7, which may miss it;
- the out-of-scope questions of exercise 3.2.

In the graph a `no` costs one more search. Count where each prompt says `no` to chunks that hold the expected file, and `yes` to an out-of-scope question.
Which grade would you put in the graph?

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Which grade would you put in the graph, and what does each wrong verdict cost?*

</div>